In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate

# Force CPU — more stable than MPS for BERT-size models on M3
device = torch.device("cpu")
print(f"Training on: {device}")

Training on: cpu


Full dataset (the updated one with 25k samples)

In [2]:
# dataset = load_dataset("imdb")

# # Use small slice first to verify pipeline works
# small_train = dataset["train"].shuffle(seed=42).select(range(2000))
# small_test  = dataset["test"].shuffle(seed=42).select(range(500))

# tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# def tokenize(batch):
#     return tokenizer(
#         batch["text"],
#         padding="max_length",
#         truncation=True,
#         max_length=256
#     )

# print("Tokenizing...")
# train_tok = small_train.map(tokenize, batched=True)
# test_tok  = small_test.map(tokenize, batched=True)

# train_tok.set_format("torch", columns=["input_ids", "attention_mask", "label"])
# test_tok.set_format("torch",  columns=["input_ids", "attention_mask", "label"])
# print("Done!")

dataset = load_dataset("imdb")

# Full dataset this time
full_train = dataset["train"]
full_test  = dataset["test"]

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

print("Tokenizing full dataset (25k train, 25k test)...")
train_tok = full_train.map(tokenize, batched=True)
test_tok  = full_test.map(tokenize, batched=True)

train_tok.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_tok.set_format("torch",  columns=["input_ids", "attention_mask", "label"])
print("Done!")

Tokenizing full dataset (25k train, 25k test)...
Done!


Load DistilBERT model

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters    : 66,955,010
Trainable parameters: 66,955,010


Metrics function

In [4]:
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1       = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {**accuracy, **f1}

Training arguments

In [5]:
training_args = TrainingArguments(
    output_dir                  = "../models/distilbert-sentiment",
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    logging_steps               = 50,
    report_to                   = "none",
    use_cpu                     = True    # ← renamed from no_cuda
)
print("Ready to train on CPU!")

Ready to train on CPU!


In [6]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = test_tok,
    compute_metrics = compute_metrics,
)

print("Training on 25k samples — let this run overnight (~3-4 hours on CPU)...")
trainer.train()

Training on 25k samples — let this run overnight (~3-4 hours on CPU)...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.253968,0.249937,0.898280,0.898005
2,0.178428,0.280473,0.910280,0.910265
3,0.092506,0.343893,0.911880,0.911876


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4689, training_loss=0.19232370553939016, metrics={'train_runtime': 13157.3686, 'train_samples_per_second': 5.7, 'train_steps_per_second': 0.356, 'total_flos': 4967527449600000.0, 'train_loss': 0.19232370553939016, 'epoch': 3.0})

In [7]:
# Cell 7 — Save the model
trainer.save_model("../models/distilbert-sentiment-final")
tokenizer.save_pretrained("../models/distilbert-sentiment-final")
print("✅ Model saved to ../models/distilbert-sentiment-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ../models/distilbert-sentiment-final


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

model_path = "/Users/samboa/sentiment-analyzer/models/distilbert-sentiment-final"
tokenizer  = AutoTokenizer.from_pretrained(model_path)
model      = AutoModelForSequenceClassification.from_pretrained(model_path)

print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

tests = [
    ("I loved this movie! Absolutely fantastic!", "should be POSITIVE"),
    ("Terrible film. I hated every minute.",      "should be NEGATIVE"),
    ("This movie is bad. I hated it.",            "should be NEGATIVE"),
]

for text, expected in tests:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1)
    print(f"\nText     : {text}")
    print(f"Expected : {expected}")
    print(f"LABEL_0  : {probs[0][0]:.4f}")
    print(f"LABEL_1  : {probs[0][1]:.4f}")
    print(f"Predicted: LABEL_{probs[0].argmax().item()} with {probs[0].max():.4f} confidence")

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './models/distilbert-sentiment-final'. Use `repo_type` argument if needed.